# AlphaFold 3 Structure Prediction on Google Cloud with `foldrun-shim`

This notebook provides an end-to-end walkthrough for running **AlphaFold 3 (Model Garden `publishers/google/models/1000376@v3_0_4`)** on **FoldRun** via **`foldrun-shim`**.

With `foldrun-shim`, both CLI commands (`!alphafold3 --json_path=...`) and Python notebook calls (`from foldrun_shim import alphafold3; alphafold3.predict(...)`) run on **Vertex AI Model Garden scale-to-zero A3 H100 GPU endpoints**, backed by Google Cloud Storage persistence and the interactive FoldRun 3D Viewer.

> **Usage Notice**: AlphaFold 3 model outputs are subject to the AlphaFold 3 Output Terms of Use (CC BY-NC-SA 4.0, non-commercial research use).

---

## AlphaFold 3 Architecture & Capabilities

- **All-Atom Prediction**: Proteins, nucleic acids (RNA, DNA), chemical components / ligands (CCD & SMILES), ions, and post-translational modifications.
- **Diffusion-Based Architecture**: Replaces AlphaFold 2's structure module with an all-atom 3D diffusion module.
- **Scale-to-Zero Economics**: Deployed on dedicated `a3-highgpu-1g` (`NVIDIA_H100_80GB`) instances that automatically scale to 0 replicas when idle (`idleScaledownPeriod: 300s`), eliminating idle GPU cost.

## 1. Prerequisites, Authentication & Installing `foldrun-shim` from GCS

1. **Authenticate to Google Cloud**:
   - **Google Colab**: Run `from google.colab import auth; auth.authenticate_user()`.
   - **Vertex AI Workbench / Colab Enterprise**: Authenticated automatically via attached service account.
   - **Local Jupyter Kernel**: Authenticate with `gcloud auth login`.
2. **Load Environment Configuration (`.env`)**:
   - The cell below loads `.env` or queries the deployment configuration.
3. **Download & Install `foldrun-shim` from GCS**:
   ```bash
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/latest_build /tmp/
   WHL=$(cat /tmp/latest_build)
   gcloud storage cp gs://${FOLDRUN_GCS_BUCKET}/sdk/${WHL} /tmp/
   pip install -q /tmp/${WHL}
   ```

In [ ]:
# @title 1. Authenticate to Google Cloud & Install `foldrun-shim` from GCS
# @markdown Authenticate first, query `latest_build` from `gs://<PROJECT_ID>-foldrun-data/sdk/`, download the newest wheel, and configure `FOLDRUN_GATEWAY_URL`. **No local GPU is required.**

import os
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def _ensure_scripts_on_path() -> None:
    """Ensure pip console scripts (including --user installs) are discoverable on PATH."""
    script_dirs: list[str] = []
    for scheme in (None, f"{os.name}_user"):
        try:
            p = (
                sysconfig.get_path("scripts")
                if scheme is None
                else sysconfig.get_path("scripts", scheme)
            )
            if p:
                script_dirs.append(p)
        except Exception:
            pass
    script_dirs.append(str(Path(sys.executable).parent))
    current_parts = os.environ.get("PATH", "").split(os.pathsep) if os.environ.get("PATH") else []
    for d in reversed(script_dirs):
        if d and d not in current_parts:
            current_parts.insert(0, d)
    os.environ["PATH"] = os.pathsep.join(current_parts)


def _is_placeholder_value(val: str) -> bool:
    lowered = (val or "").strip().lower()
    return not lowered or any(
        tok in lowered
        for tok in (
            "your-project-foldrun-data",
            "your-project",
            "your_project",
            "example",
            "<",
        )
    )


def _is_placeholder_bucket(bucket: str) -> bool:
    clean = (bucket or "").strip().removeprefix("gs://").rstrip("/")
    return _is_placeholder_value(clean)


_ensure_scripts_on_path()


def _load_foldrun_dotenv(custom_path: Path | None = None) -> Path | None:
    """Load key=value pairs from a local .env file generated by deploy-all.sh."""
    if custom_path is not None:
        candidates = [custom_path]
    else:
        candidates = []
        if os.environ.get("FOLDRUN_ENV_FILE"):
            candidates.append(Path(os.environ["FOLDRUN_ENV_FILE"]))
        candidates.extend(
            [Path(".env"), Path("../.env"), Path("../../.env"), Path("/content/.env")]
        )
    loaded_from: Path | None = None
    for path in candidates:
        if not path.is_file():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = re.sub(r"^export(?=\s)\s*", "", k.strip())
            v = v.strip().strip("'").strip('"')
            if k and v:
                existing = os.environ.get(k, "")
                if not existing or _is_placeholder_value(existing):
                    os.environ[k] = v
        if loaded_from is None:
            loaded_from = path.resolve()
        if custom_path is not None or not _is_placeholder_value(
            os.environ.get("FOLDRUN_GATEWAY_URL", "")
        ):
            return path.resolve()
    return loaded_from


_dotenv_path = _load_foldrun_dotenv()
if _dotenv_path:
    print(f"Loaded FoldRun configuration from {_dotenv_path}")

FOLDRUN_GCS_BUCKET = "your-project-foldrun-data"  # @param {type:"string"}
FOLDRUN_GATEWAY_URL = "https://foldrun-gateway-example-uc.a.run.app"  # @param {type:"string"}
FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT = ""  # @param {type:"string"}

if not _is_placeholder_bucket(FOLDRUN_GCS_BUCKET):
    os.environ["FOLDRUN_GCS_BUCKET"] = FOLDRUN_GCS_BUCKET.strip()
else:
    os.environ.setdefault("FOLDRUN_GCS_BUCKET", "your-project-foldrun-data")

if not _is_placeholder_value(FOLDRUN_GATEWAY_URL):
    os.environ["FOLDRUN_GATEWAY_URL"] = FOLDRUN_GATEWAY_URL.strip()
else:
    os.environ.setdefault("FOLDRUN_GATEWAY_URL", "https://foldrun-gateway-example-uc.a.run.app")

if FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT.strip() and not _is_placeholder_value(
    FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT
):
    os.environ["FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT"] = FOLDRUN_IMPERSONATE_SERVICE_ACCOUNT.strip()

# 1. Authenticate to Google Cloud first (required in standard Google Colab before accessing GCS)
if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()


def _install_foldrun_shim_from_gcs(gcs_bucket: str) -> None:
    """Download latest foldrun_shim wheel via latest_build from gs://<bucket>/sdk/ and install via pip."""
    clean_bucket = (gcs_bucket or "").strip().removeprefix("gs://").rstrip("/")
    bucket_configured = not _is_placeholder_bucket(clean_bucket)

    if bucket_configured and _is_placeholder_value(os.environ.get("FOLDRUN_GATEWAY_URL", "")):
        try:
            env_dst = Path("/tmp/foldrun_sdk.env")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/.env",
                    str(env_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and env_dst.is_file():
                if _load_foldrun_dotenv(env_dst):
                    print(f"Loaded remote FoldRun configuration from gs://{clean_bucket}/sdk/.env")
        except Exception as exc:
            print(f"Notice: could not load gs://{clean_bucket}/sdk/.env: {exc}")

    whl_filename = "foldrun_shim-0.1.0-py3-none-any.whl"
    if bucket_configured:
        try:
            latest_build_dst = Path("/tmp/foldrun_latest_build")
            res = subprocess.run(
                [
                    "gcloud",
                    "storage",
                    "cp",
                    f"gs://{clean_bucket}/sdk/latest_build",
                    str(latest_build_dst),
                ],
                check=False,
                capture_output=True,
                text=True,
            )
            if res.returncode == 0 and latest_build_dst.is_file():
                candidate = latest_build_dst.read_text(encoding="utf-8").strip()
                if candidate.endswith(".whl"):
                    whl_filename = candidate
                    print(
                        f"Detected latest FoldRun SDK build from gs://{clean_bucket}/sdk/latest_build: {whl_filename}"
                    )
        except Exception as exc:
            print(f"Notice: could not read gs://{clean_bucket}/sdk/latest_build: {exc}")

    whl_path = Path(f"/tmp/{whl_filename}")
    marker_path = Path(f"/tmp/{whl_filename}.installed")
    if marker_path.exists():
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            return
        except ImportError:
            pass

    if not bucket_configured:
        try:
            import foldrun_shim

            _ensure_scripts_on_path()
            print(
                "Notice: FOLDRUN_GCS_BUCKET is unconfigured; using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"foldrun_shim is not installed. Set FOLDRUN_GCS_BUCKET='<PROJECT_ID>-foldrun-data' "
                f"so this cell can download gs://<PROJECT_ID>-foldrun-data/sdk/{whl_filename}."
            )

    gcs_uri = f"gs://{clean_bucket}/sdk/{whl_filename}"
    print(f"Downloading {whl_filename} from {gcs_uri}...")
    cp_res = subprocess.run(
        ["gcloud", "storage", "cp", gcs_uri, str(whl_path)],
        check=False,
        capture_output=True,
        text=True,
    )
    if cp_res.returncode != 0:
        err_line = (
            (cp_res.stderr or cp_res.stdout or "gcloud storage cp failed").strip().splitlines()[-1]
        )
        try:
            import foldrun_shim  # noqa: F401

            _ensure_scripts_on_path()
            print(
                f"Warning: could not download {gcs_uri} ({err_line}); using locally installed foldrun_shim."
            )
            return
        except ImportError:
            raise RuntimeError(
                f"Failed to download {gcs_uri} ({err_line}) and foldrun_shim is not installed locally."
            )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(whl_path)],
        check=True,
    )
    _ensure_scripts_on_path()
    marker_path.write_text("installed", encoding="utf-8")
    print(f"Installed {whl_filename} successfully.")


_install_foldrun_shim_from_gcs(os.environ.get("FOLDRUN_GCS_BUCKET", FOLDRUN_GCS_BUCKET))
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "py3Dmol",
        "matplotlib",
        "numpy",
        "pyyaml",
    ],
    check=False,
)

import foldrun_shim
from foldrun_shim import alphafold3

print(f"foldrun-shim version : {getattr(foldrun_shim, '__version__', 'unknown')}")
print(f"alphafold3 in PATH   : {shutil.which('alphafold3') or 'available via foldrun_shim.cli'}")
print(f"FOLDRUN_GATEWAY_URL  : {os.environ['FOLDRUN_GATEWAY_URL']}")

In [ ]:
# @title 1B. (Optional) Check Live GPU Availability & Configure On-Demand vs. DWS Scheduling
# @markdown Thin-permission notebook users (who only have `roles/run.invoker` on the FoldRun Gateway)
# @markdown can inspect live GPU availability, active/queued jobs, and regional quotas directly from Python
# @markdown without needing GCP project-level GCE or ServiceUsage IAM permissions.
# @markdown
# @markdown > **Note for AlphaFold 3:** AlphaFold 3 inference executes on the dedicated scale-to-zero Vertex AI
# @markdown > Model Garden endpoint (`foldrun-alphafold3-ep`, `a3-highgpu-1g` / `NVIDIA_H100_80GB`).
# @markdown
# @markdown **CLI Equivalents:**
# @markdown - `!foldrun gpus` (inspect live GPU availability and queue depths)
# @markdown - `!foldrun status <job_id>` (inspect status of an active or queued job)
# @markdown - `!foldrun cancel <job_id>` (cancel a queued or running job)
# @markdown - GCP Project Admins only: `!gcloud alpha services quota list --service=aiplatform.googleapis.com --consumer=projects/$GCP_PROJECT_ID`

import os

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. Query live GPU availability and queue status via FoldRun Gateway
try:
    import foldrun_shim

    gpu_status = foldrun_shim.check_gpus()
    display(gpu_status)
except Exception as exc:
    print(f"Notice: Could not query live GPU status via FoldRun Gateway: {exc}")
    print(
        "You can still proceed with prediction using default scheduling or environment variables below."
    )

## Part A — Upstream Input Format & Verbatim CLI Prediction

AlphaFold 3 accepts standard JSON inputs conforming to the `alphafold3` dialect with protein, RNA, DNA, or ligand chains.
The cell below generates an example query for Human Troponin C (`P09496`, essential calcium-binding muscle protein).

In [ ]:
import json

# Note: AlphaFold 3 requires explicit empty `unpairedMsa`, `pairedMsa`, and `templates`
# fields on protein chains when running inference-only with `--run_data_pipeline=false`.
troponin_query = {
    "name": "troponin_c_human",
    "modelSeeds": [1],
    "sequences": [
        {
            "protein": {
                "id": "A",
                "sequence": "MDDIYKAAVEQLTEEQKNEFKAAFDIFVLGAEDGCISTKELGKVMRMLGQNPTPEELQEMIDEVDEDGSGTVDFDEFLVMMVRCMKDDSKGKSEEELSDLFRMFDKNADGYIDLDELKIMLQATGETITEDDIEELMKDGDKNNDGRIDYDEFLEFMKGVE",
                "unpairedMsa": "",
                "pairedMsa": "",
                "templates": [],
            }
        }
    ],
    "dialect": "alphafold3",
    "version": 1,
}

with open("troponin_c.json", "w") as f:
    json.dump(troponin_query, f, indent=2)

print(
    "Wrote troponin_c.json successfully (with explicit MSA-free fields for --run_data_pipeline=false)."
)

### Run Prediction via Native CLI Shim

You can invoke `alphafold3` directly from the shell or terminal. The CLI intercepts the call, packages inputs, submits to `foldrun-gateway`, tracks progress, and downloads outputs.

In [ ]:
# Execute AlphaFold 3 via CLI shim (fast MSA-free test with 1 diffusion sample)
!alphafold3 --json_path=troponin_c.json --output_dir=./af3_troponin_results --num_diffusion_samples=1 --run_data_pipeline=false

## Part B — FoldRun Python API (`foldrun_shim.alphafold3`)

The Python helper provides seamless programmatic execution with state persistence and HTML rendering in Jupyter.

In [ ]:
# Run prediction programmatically using the Python helper
result = alphafold3.predict(
    json_path="troponin_c.json",
    output_dir="./af3_troponin_results",
    num_diffusion_samples=1,
    run_data_pipeline=False,
)

# Display rich job status card with link to 3D Viewer
result

## Part C — Inspect Generated Structures and Metrics

Once prediction finishes, `result.extracted_files` contains the materialized outputs, including:
- `*_model.cif`: Top predicted 3D all-atom structure in mmCIF format.
- `summary_confidences_*.json`: Overall pLDDT, pTM, ipTM, and per-chain confidence metrics.
- `confidences_*.json`: Per-residue pLDDT and per-token predicted aligned error (PAE) matrices.

In [ ]:
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

out_dir = Path("./af3_troponin_results")
out_dir.mkdir(parents=True, exist_ok=True)

materialized_files = sorted(
    p for p in out_dir.rglob("*") if p.is_file() and p.name != ".foldrun_job.json"
)
if materialized_files:
    print(f"Files materialized in {out_dir}:")
    for p in materialized_files:
        rel_name = p.relative_to(out_dir).as_posix()
        size_kb = p.stat().st_size / 1024
        print(f" - {rel_name:<45} ({size_kb:8.1f} KB)")

summary_files = sorted(out_dir.rglob("*summary*confidences*.json")) + sorted(
    out_dir.rglob("summary_metrics.json")
)
conf_files = sorted(
    p
    for p in out_dir.rglob("*confidences*.json")
    if "summary" not in p.name and p.name != "summary_metrics.json"
) + sorted(out_dir.rglob("confidence_json.json"))
cif_files = sorted(out_dir.rglob("*.cif"))

synthetic_demo_mode = os.environ.get("FOLDRUN_SYNTHETIC_DEMO", "").strip().lower() == "true"
using_synthetic = False

if not cif_files and not conf_files:
    actual_list = [p.relative_to(out_dir).as_posix() for p in materialized_files] or ["<empty>"]
    if not synthetic_demo_mode:
        raise FileNotFoundError(
            f"No AlphaFold 3 prediction outputs (*.cif or *confidences*.json) found in {out_dir.resolve()}.\n"
            f"Files currently in {out_dir}: {actual_list}\n"
            f"Run the AlphaFold 3 prediction cell above against a live FoldRun Gateway first "
            f"(or set FOLDRUN_SYNTHETIC_DEMO=true strictly to test notebook chart layout offline)."
        )
    using_synthetic = True
    print("=" * 78)
    print("SYNTHETIC DEMO DATA (NOT AN ALPHAFOLD 3 PREDICTION) — FOLDRUN_SYNTHETIC_DEMO=true")
    print("=" * 78)

if summary_files:
    summary_data = json.loads(summary_files[0].read_text(encoding="utf-8"))
elif using_synthetic:
    summary_data = {
        "ranking_score": 0.89,
        "ptm": 0.88,
        "iptm": None,
        "fraction_disordered": 0.04,
    }
else:
    summary_data = {}

if conf_files:
    conf_data = json.loads(conf_files[0].read_text(encoding="utf-8"))
else:
    rng = np.random.default_rng(42)
    n_res = 76
    atom_plddts = np.clip(
        88.0 + 6.0 * np.sin(np.linspace(0, 4 * np.pi, n_res)), 55.0, 98.5
    ).tolist()
    grid = np.abs(np.subtract.outer(np.arange(n_res), np.arange(n_res)))
    pae_mat = np.clip(1.5 + 0.12 * grid + rng.normal(0, 0.4, (n_res, n_res)), 0.5, 25.0).tolist()
    conf_data = {"atom_plddts": atom_plddts, "pae": pae_mat}

print(
    f"\nSummary Metrics -> ranking_score={summary_data.get('ranking_score')}, "
    f"pTM={summary_data.get('ptm')}, ipTM={summary_data.get('iptm')}"
)

plddt_vals = np.array(conf_data.get("atom_plddts") or conf_data.get("plddt") or [], dtype=float)
pae_vals = np.array(conf_data.get("pae") or np.zeros((1, 1)), dtype=float)

title_prefix = "[SYNTHETIC DEMO DATA — NOT AN AF3 PREDICTION] " if using_synthetic else ""
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
if using_synthetic:
    fig.suptitle(
        "SYNTHETIC DEMO DATA (NOT AN ALPHAFOLD 3 PREDICTION)",
        color="#b31412",
        fontweight="bold",
    )

if len(plddt_vals) > 0:
    axes[0].plot(np.arange(1, len(plddt_vals) + 1), plddt_vals, color="#1a73e8", lw=1.8)
axes[0].axhline(90, color="#0d652d", ls="--", lw=0.9, label="Very high (>=90)")
axes[0].axhline(70, color="#f9ab00", ls="--", lw=0.9, label="Confident (>=70)")
axes[0].set_ylim(0, 100)
axes[0].set_xlabel("Atom / Token Index")
axes[0].set_ylabel("pLDDT")
axes[0].set_title(f"{title_prefix}AlphaFold 3 Predicted Local Confidence (pLDDT)")
axes[0].legend(loc="lower right", fontsize=8)
axes[0].grid(alpha=0.25)

im = axes[1].imshow(pae_vals, cmap="Greens_r", vmin=0, vmax=31.75, origin="upper")
axes[1].set_title(f"{title_prefix}Predicted Aligned Error (PAE, Å)")
axes[1].set_xlabel("Scored Residue")
axes[1].set_ylabel("Aligned Residue")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="Expected Position Error (Å)")
plt.tight_layout()
plt.show()

if cif_files or using_synthetic:
    try:
        import py3Dmol

        view = py3Dmol.view(width=760, height=480)
        if cif_files:
            view.addModel(cif_files[0].read_text(encoding="utf-8"), "cif")
        else:
            demo_cif_lines = [
                "data_SYNTHETIC_DEMO",
                "#",
                "loop_",
                "_atom_site.group_PDB",
                "_atom_site.id",
                "_atom_site.type_symbol",
                "_atom_site.label_atom_id",
                "_atom_site.label_alt_id",
                "_atom_site.label_comp_id",
                "_atom_site.label_asym_id",
                "_atom_site.label_entity_id",
                "_atom_site.label_seq_id",
                "_atom_site.pdbx_PDB_ins_code",
                "_atom_site.Cartn_x",
                "_atom_site.Cartn_y",
                "_atom_site.Cartn_z",
                "_atom_site.occupancy",
                "_atom_site.B_iso_or_equiv",
                "_atom_site.auth_seq_id",
                "_atom_site.auth_comp_id",
                "_atom_site.auth_asym_id",
                "_atom_site.pdbx_PDB_model_num",
            ]
            for idx in range(36):
                ang = idx * (2.0 * math.pi / 3.6)
                x, y, z = (
                    round(5.5 * math.cos(ang), 3),
                    round(5.5 * math.sin(ang), 3),
                    round(idx * 1.5, 3),
                )
                demo_cif_lines.append(
                    f"ATOM {idx + 1} C CA . ALA A 1 {idx + 1} ? {x} {y} {z} 1.00 85.00 {idx + 1} ALA A 1"
                )
            demo_cif_lines.append("#")
            view.addModel("\n".join(demo_cif_lines) + "\n", "cif")
        view.setStyle(
            {
                "cartoon": {
                    "colorscheme": {
                        "prop": "b",
                        "gradient": "roygb",
                        "min": 50,
                        "max": 90,
                    }
                }
            }
        )
        view.zoomTo()
        view.show()
    except Exception as viz_exc:
        print(f"3D visualization note: {viz_exc}")